# Lab 3 · Neural network: recognise handwritten digits
**Goal:** read a handwritten digit (0–9) from an 8×8 image, and know when not to trust the answer.

**You will practise**
- Scale inputs; train / validation / test split
- Start from a simple baseline
- Build, compile and train a Keras model
- Read learning curves; spot overfitting
- Fix it: early stopping, dropout
- Use a model built for images (CNN)
- Compare on validation; test once
- Read the mistakes; use confidence to hand hard cases to a human
- Save, reload, predict

**How to work:** 🧠 = write your guess first · ✓ = the notebook proves the result

## 0 · Setup
- Fix all random seeds (NumPy, Python, TensorFlow) with one call
- Small data: runs on a laptop CPU in about a minute

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

SEED = 0
keras.utils.set_random_seed(SEED)
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "font.size": 12})
print("keras", keras.__version__, "· backend", keras.backend.backend())

## 1 · Load and look
- 1,797 images, 8×8 pixels, values 0–16
- Target: the digit 0–9

In [ ]:
digits = load_digits()
images, labels = digits.images, digits.target
print(images.shape, "· classes:", np.bincount(labels))
fig, axes = plt.subplots(2, 10, figsize=(12, 2.8))
for ax, img, lab in zip(axes.ravel(), images, labels):
    ax.imshow(img, cmap="gray_r"); ax.set_title(lab); ax.axis("off")
plt.tight_layout(); plt.show()

**What to notice**
- Classes are balanced (≈ 180 each): accuracy is a fair score here
- Low resolution: some 1s, 7s and 9s look alike

## 2 · Scale and split
- Scale pixels to 0–1 (neural networks train best on small, similar-sized inputs)
- Three piles, stratified: train 60% · validation 20% (compare, tune, stop) · test 20% (locked, used once)

In [ ]:
X = (images / 16.0).astype("float32")
y = labels
X_trval, X_test, y_trval, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_trval, y_trval, test_size=0.25, stratify=y_trval, random_state=SEED)
print(f"train {len(X_train)} · validation {len(X_val)} · test {len(X_test)} (locked)")

## 3 · 🧠 Predict first
- Logistic regression on the 64 raw pixels: what validation accuracy?
- Can a neural network do better?

In [ ]:
my_guess = {"logistic": None, "best_network": None}

## 4 · Baseline: logistic regression
- 64 pixels in a row, one weight per pixel per digit
- Every network must beat this

In [ ]:
flat = lambda a: a.reshape(len(a), -1)
logistic = LogisticRegression(max_iter=2000).fit(flat(X_train), y_train)
baseline_acc = logistic.score(flat(X_val), y_val)
print(f"logistic regression: validation accuracy {baseline_acc:.1%}")

## 5 · A first network
- Flatten → Dense(64, ReLU) → Dense(10, softmax)
- Loss: sparse categorical cross-entropy (labels are integers 0–9)
- Optimiser: Adam, learning rate 0.001
- Always pass validation data to `fit` to watch it learn

In [ ]:
def train(model, epochs, callbacks=(), name=""):
    """Compile, train with validation, return the history as a DataFrame."""
    keras.utils.set_random_seed(SEED)
    # TODO: model.compile(Adam(1e-3), sparse_categorical_crossentropy, metrics=["accuracy"])
    # TODO: h = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=epochs, batch_size=32, verbose=0, callbacks=list(callbacks))
    ...

# TODO: small = Sequential([Input((8, 8)), Flatten(), Dense(64, relu), Dense(10, softmax)]); hist_small = train(small, 100)

## 6 · Learning curves
- Training loss should fall
- Validation loss falls, then flattens: that's where learning stops paying off

In [ ]:
def plot_curves(hist, title):
    fig, (a, b) = plt.subplots(1, 2, figsize=(12, 3.8))
    hist[["loss", "val_loss"]].plot(ax=a, title=f"{title}: loss"); a.set_xlabel("epoch")
    hist[["accuracy", "val_accuracy"]].plot(ax=b, title=f"{title}: accuracy", ylim=(0.8, 1.01)); b.set_xlabel("epoch")
    plt.tight_layout(); plt.show()

plot_curves(hist_small, "small dense")

## 7 · 🧠 A much bigger network, trained for longer
- Two layers of 512 neurons (≈ 300,000 weights for 1,077 images), 200 epochs, no protection
- What happens to the validation loss?

In [ ]:
def big_dense(dropout=0.0):
    layers = [keras.Input((8, 8)), keras.layers.Flatten()]
    for _ in range(2):
        layers.append(keras.layers.Dense(512, activation="relu"))
        if dropout: layers.append(keras.layers.Dropout(dropout))
    return keras.Sequential(layers + [keras.layers.Dense(10, activation="softmax")])

hist_big = train(big_dense(), 200, name="big, no protection")
plot_curves(hist_big, "big, no protection")

In [ ]:
best_epoch = int(hist_big.val_loss.idxmin())
print(f"val loss: best {hist_big.val_loss.min():.3f} at epoch {best_epoch} → {hist_big.val_loss.iloc[-1]:.3f} at the end · train accuracy {hist_big.accuracy.iloc[-1]:.0%}")
assert hist_big.val_loss.iloc[-1] > 1.3 * hist_big.val_loss.min() and hist_big.accuracy.iloc[-1] == 1.0
print("✓ overfitting: training is perfect while validation loss climbs after its best epoch")

**Overfitting**
- The network memorises the training images
- Validation loss climbs back up, even while accuracy looks fine
- Watch the **loss**, not only accuracy: it shows the network growing over-confident

## 8 · Fix it: early stopping and dropout
- **Early stopping:** stop when validation loss hasn't improved for 15 epochs; keep the best weights
- **Dropout:** switch off a random 40% of neurons at every training step, so no neuron can be relied on alone

In [ ]:
stop = lambda: keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)
# TODO: train big_dense() with [stop()] for up to 300 epochs → hist_es
# TODO: train big_dense(dropout=0.4) with [stop()] → hist_drop

In [ ]:
assert len(hist_es) < 100 and hist_drop.val_loss.min() < hist_big.val_loss.min()
print(f"✓ early stopping ended after {len(hist_es)} of 300 epochs; dropout reached a lower validation loss ({hist_drop.val_loss.min():.3f} vs {hist_big.val_loss.min():.3f})")

## 9 · A network built for images: CNN
- **Convolution:** small 3×3 filters slide over the image and detect strokes and corners anywhere
- **Pooling:** shrink the map, keep the strongest signals
- Far fewer weights than the big dense network, because each filter is reused everywhere

In [ ]:
def cnn():
    return keras.Sequential([
        keras.Input((8, 8, 1)),
        keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
        keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
        keras.layers.MaxPooling2D(),
        keras.layers.Flatten(),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(10, activation="softmax"),
    ], name="cnn")

hist_cnn = train(cnn(), 300, [stop()], name="CNN + dropout + early stopping")
plot_curves(hist_cnn, "CNN")

## 10 · Compare on validation
- Accuracy differences are small; validation **loss** shows how confident and right each model is

In [ ]:
final_val = lambda h: h.loc[h.val_loss.idxmin()]
summary = pd.DataFrame({
    "logistic regression": {"val accuracy": baseline_acc, "val loss": np.nan},
    "small dense": {"val accuracy": final_val(hist_small).val_accuracy, "val loss": hist_small.val_loss.min()},
    "big, no protection": {"val accuracy": hist_big.val_accuracy.iloc[-1], "val loss": hist_big.val_loss.iloc[-1]},
    "big + dropout + ES": {"val accuracy": final_val(hist_drop).val_accuracy, "val loss": hist_drop.val_loss.min()},
    "CNN": {"val accuracy": final_val(hist_cnn).val_accuracy, "val loss": hist_cnn.val_loss.min()},
}).T
summary.round(3)

In [ ]:
assert summary["val loss"].idxmin() == "CNN" and summary.loc["CNN", "val accuracy"] > baseline_acc
print(f"✓ the CNN wins: lowest validation loss, and {summary.loc['CNN', 'val accuracy']:.1%} vs {baseline_acc:.1%} for logistic regression")

## 11 · Final test: once
- Retrain the chosen CNN exactly as before (same seed, same early stopping)
- Score the locked test set one time; give a range

In [ ]:
keras.utils.set_random_seed(SEED)
model = cnn()
_ = train(model, 300, [stop()], name="final CNN")
probs = model.predict(X_test, verbose=0)
pred = probs.argmax(axis=1)
correct = pred == y_test

rng = np.random.default_rng(SEED)
boot = [correct[rng.integers(0, len(correct), len(correct))].mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"test accuracy {correct.mean():.1%} (95% range {lo:.1%}–{hi:.1%}) · {(~correct).sum()} mistakes of {len(y_test)}")
assert correct.mean() >= 0.97
print("✓ proof: the CNN reads at least 97% of digits it has never seen")

## 12 · Read the mistakes
- Which digits get confused?
- Look at the images it got wrong: would you have got them right?

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, pred, cmap="Blues", colorbar=False); plt.title("Test set"); plt.show()
wrong = np.where(~correct)[0]
fig, axes = plt.subplots(1, len(wrong), figsize=(1.6 * len(wrong), 2.2))
for ax, i in zip(np.atleast_1d(axes), wrong):
    ax.imshow(X_test[i], cmap="gray_r"); ax.set_title(f"true {y_test[i]}\nsaid {pred[i]} ({probs[i].max():.0%})", fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()

## 13 · 🧠 Use confidence: let a human check the hard ones
- The softmax output is a probability for each digit
- Rule: accept the answer only if its probability ≥ 0.9; otherwise send to a person
- Predict: what share needs a person? How accurate are the accepted ones?

In [ ]:
THRESHOLD = 0.9
confident = probs.max(axis=1) >= THRESHOLD
print(f"sent to a person: {(~confident).mean():.1%} · accuracy on the rest: {correct[confident].mean():.1%} (all: {correct.mean():.1%})")
print(f"mistakes caught by the rule: {(~correct & ~confident).sum()} of {(~correct).sum()}")
assert correct[confident].mean() >= correct.mean()
print("✓ low-confidence answers hold most of the mistakes: checking a few cases buys a lot of accuracy")

## 14 · Save, reload, predict
- Save the whole model (architecture + weights) in one `.keras` file
- Reload and check it gives identical predictions

In [ ]:
Path("models").mkdir(exist_ok=True)
model.save("models/digits_cnn.keras")
loaded = keras.models.load_model("models/digits_cnn.keras")
assert np.allclose(loaded.predict(X_test, verbose=0), probs, atol=1e-6)

def read_digit(image_0_to_16):
    """8×8 array with values 0–16 → (digit, probability, needs_human)."""
    p = loaded.predict((np.asarray(image_0_to_16, "float32") / 16.0)[None], verbose=0)[0]
    return int(p.argmax()), round(float(p.max()), 3), bool(p.max() < THRESHOLD)

print("first test image:", read_digit(X_test[0] * 16), "· true", y_test[0])
print("✓ the reloaded model gives identical predictions")

## 15 · Your turn
- Change the dropout rate to 0 and 0.6 in the CNN: what happens to validation loss?
- Try learning rate 0.01 and 0.0001: how do the learning curves change?
- Shift every test image one pixel right (`np.roll(X_test, 1, axis=2)`): which drops more, the CNN or the small dense network?

In [ ]:
# TODO

## ✅ Checklist for every neural-network project
- [x] Scaled inputs; split train / validation / test, stratified
- [x] Beat a simple baseline first
- [x] Watched learning curves (loss, not only accuracy)
- [x] Recognised overfitting; used early stopping and dropout
- [x] Picked an architecture that fits the data (CNN for images)
- [x] Compared on validation; tested once, with a range
- [x] Read the mistakes
- [x] Used confidence to route hard cases to a person
- [x] Fixed seeds; saved and reloaded the model, checked identical output